# P2.6 · Small CNN on CIFAR-10 (conv/pool/BN, augmentation, OneCycle, early stopping)

**ID:** P2.6 · **Target:** 35 min · **Plan week:** 3 (timed repeat in week 5) · **Part:** P (PyTorch first principles)

**The drill.** Train a small VGG-style CNN on CIFAR-10 (32×32 RGB, 10 classes, 50k train / 10k test) to **≥ 80 % test
accuracy within a few minutes on one GPU**. Hold out a val split, augment on the fly (random crop with padding +
horizontal flip, **train only**), use a one-cycle LR schedule, and early-stop / checkpoint on val.

**Must include (from the plan):** conv/pool/BN, augmentation, LR schedule, early stopping. Also: per-channel
normalization with train statistics, initial-loss check (≈ ln 10), overfit-one-batch, whole-set eval under
`eval()` + `no_grad()`, test touched once.

**Speed design (why it trains in ~1 min, not ~20).** Following Keller Jordan's airbench, the whole dataset lives on
the GPU as one tensor and augmentation is done **per batch on the GPU** (no `DataLoader` workers decoding PIL
images). Training uses bf16 autocast and `channels_last`. The net is conv-BN-ReLU blocks with max-pool (as in airbench's
`ConvGroup`), a global max-pool, and a final logit scale of 1/8 (airbench uses 1/9). The scale shrinks the logits and
their gradients under the large peak LR. It is optional: the measured initial loss is ≈ ln 10 with or without it.
SGD + Nesterov with OneCycle (Smith & Topin) and label smoothing 0.1 (airbench uses 0.2). The
torchvision-transforms + `DataLoader(num_workers=…)` route from the PyTorch tutorial is also fine in an interview.
Say why you chose one or the other.

**Rules.** No docs, no LLM. Data: `torchvision.datasets.CIFAR10(DATA, download=True)` (already in `DATA`; the call
only verifies the md5). Each solution cell keeps only its spec. The self-check names the variables it expects.

## Sources
- A. Krizhevsky, *Learning Multiple Layers of Features from Tiny Images*, Tech. report, U. Toronto, 2009 —
  https://www.cs.toronto.edu/~kriz/learning-features-2009-TR.pdf (CIFAR-10).
- K. Jordan, *94% on CIFAR-10 in 3.29 Seconds on a Single GPU*, 2024, arXiv:2404.00498 — https://arxiv.org/abs/2404.00498;
  code KellerJordan/cifar10-airbench @4c1b6d1, `legacy/airbench94.py`, MIT
  (https://github.com/KellerJordan/cifar10-airbench/blob/4c1b6d1e3889b037efadcfd5c0ea65b246592362/legacy/airbench94.py):
  GPU-resident loader, batch flip/crop, conv-pool-BN-GELU groups, logit scaling, label smoothing. The crop below is
  a simpler gather-based rewrite of its `batch_crop`; nothing else is copied.
- airbench cites D. Page's DAWNBench series *How to train your ResNet* (myrtle.ai, 2018), the origin of the
  GPU-loader + small-net recipe: https://web.archive.org/web/2023/https://myrtle.ai/learn/how-to-train-your-resnet/ (archived; the original URL now 404s)
- PyTorch tutorial *Training a Classifier* (CIFAR-10 with torchvision transforms and DataLoader; BSD-3):
  https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
- S. Ioffe, C. Szegedy, *Batch Normalization*, ICML 2015, arXiv:1502.03167 — https://arxiv.org/abs/1502.03167
- L. N. Smith, N. Topin, *Super-Convergence: Very Fast Training of Neural Networks Using Large Learning Rates*, 2017,
  arXiv:1708.07120 — https://arxiv.org/abs/1708.07120 (one-cycle policy; `torch.optim.lr_scheduler.OneCycleLR`:
  https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.OneCycleLR.html).
- T. He et al., *Bag of Tricks for Image Classification with Convolutional Neural Networks*, CVPR 2019,
  arXiv:1812.01187 — https://arxiv.org/abs/1812.01187 (label smoothing, warm-up, no weight decay on BN/bias).
- V. Godbole et al., *Deep Learning Tuning Playbook* (2023), "retrospective optimal checkpoint selection",
  google-research/tuning_playbook @7554a51 `README.md`, CC-BY-4.0: https://github.com/google-research/tuning_playbook

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

from prep import DATA, Session, check, check_close, seed_everything

s = Session("P2.6", target_min=35)
seed_everything(0)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
AMP = dict(device_type=dev.type, dtype=torch.bfloat16, enabled=dev.type == "cuda")
CLASSES = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

In [ ]:
# 1. DATA. CIFAR10(DATA, train=True/False, download=True). `.data` is uint8 (N, 32, 32, 3) -> float (N, 3, 32, 32) in
#    [0, 1]; `.targets` -> int64. Val = last 5000 of a seeded permutation (generator seed 0) of the 50k train images.
#    Per-CHANNEL mean/std from the 45k train images only -> `mean`, `std` of shape (1, 3, 1, 1).
#    Normalize all splits and move them to `dev`: X_train, y_train, X_val, y_val, X_test, y_test.
# YOUR CODE HERE


In [ ]:
# 2. AUGMENTATION (GPU, per batch, TRAIN ONLY). augment(x, pad=4) for x (B, 3, 32, 32):
#    reflect-pad by 4, then an independent random 32x32 crop PER IMAGE (offsets in [0, 8]), then flip each image
#    horizontally with probability 0.5. Vectorize: build row/col index grids from per-image offsets and gather with
#    advanced indexing (no Python loop over images). Show a few augmented train images.
# YOUR CODE HERE


In [ ]:
# 3. MODEL. `make_model(scale=0.125)`: nn.Sequential of conv-BN-ReLU blocks (Conv2d 3x3, padding=1, bias=False ->
#    BatchNorm2d -> ReLU), widths 64 -> 128 -> 256 -> 512, MaxPool2d(2) after the 128, 256 and 512 blocks
#    (32 -> 16 -> 8 -> 4), then AdaptiveMaxPool2d(1), Flatten, Linear(512, 10), and multiply the logits by `scale`
#    (a tiny `Scale` module). Move to `dev`, channels_last. Print the parameter count.
# YOUR CODE HERE


In [ ]:
# 4. EVAL. evaluate(model, X, y, bs=2000) -> (mean CE loss, accuracy) over the whole tensor set:
#    model.eval(), torch.no_grad(), autocast(**AMP), NO augmentation; losses summed then / N.
# YOUR CODE HERE


In [ ]:
# 5. SANITY CHECKS. (a) `init_loss` = val loss of the fresh `model` (≈ ln 10); also print it for make_model(scale=1.0)
#    (BN keeps activations O(1) and the default Linear init gives small logits, so both should be ≈ ln 10). (b) Overfit one batch: fresh model, 128 train images (no augmentation),
#    train mode, Adam lr=1e-3, plain CE (no label smoothing: its minimum is not 0), 100 steps -> `overfit_loss` < 0.05.
# YOUR CODE HERE


In [ ]:
# 6. TRAIN. epochs=18, batch 512 (shuffle with torch.randperm each epoch; drop nothing), augment every train batch.
#    Loss: CrossEntropyLoss(label_smoothing=0.1). Optimizer: SGD(lr=0.1, momentum=0.9, nesterov=True,
#    weight_decay=5e-4). Schedule: OneCycleLR(max_lr=0.2, total_steps=epochs * steps_per_epoch, pct_start=0.25),
#    stepped EVERY BATCH; log the LR per step in `lrs`. bf16 autocast, channels_last.
#    Early stopping: evaluate on val each epoch; keep the best-val-acc state_dict (deep copy) -> `best_state`,
#    `best_epoch`; stop if no improvement for `patience = 4` epochs. Record `history` with "train_loss", "val_loss",
#    "val_acc" per epoch and the wall-clock `train_seconds`. Load `best_state` at the end.
# YOUR CODE HERE


In [ ]:
# 7. TEST (once) + CURVES. `test_loss, test_acc = evaluate(model, X_test, y_test)`. Per-class test accuracy.
#    Plot: (left) train vs val loss per epoch (train loss includes label smoothing + augmentation: say why it can sit
#    ABOVE val loss), (right) the LR schedule `lrs`.
# YOUR CODE HERE


## Self-check
Expects: `mean`, `std`, `X_train`, `augment`, `model`, `make_model`, `evaluate`, `init_loss`, `overfit_loss`,
`history`, `lrs`, `best_state`, `best_acc`, `train_seconds`, `test_acc`.

In [ ]:
check_close("per-channel train mean ~ 0", X_train[:20000].mean((0, 2, 3)), torch.zeros(3), atol=0.02)
check("mean/std are per channel", tuple(mean.shape) == (1, 3, 1, 1) and tuple(std.shape) == (1, 3, 1, 1))
kinds = {type(m) for m in model.modules()}
check("model has Conv2d, BatchNorm2d and pooling", nn.Conv2d in kinds and nn.BatchNorm2d in kinds and
      any(issubclass(k, (nn.MaxPool2d, nn.AvgPool2d, nn.AdaptiveMaxPool2d, nn.AdaptiveAvgPool2d)) for k in kinds))
check("no Softmax in model", not any(issubclass(k, (nn.Softmax, nn.LogSoftmax)) for k in kinds))
xa = X_train[:1].repeat(64, 1, 1, 1)
aug = augment(xa)
check("augment keeps shape", aug.shape == xa.shape)
check("augment is random PER IMAGE", (aug - aug[:1]).abs().flatten(1).amax(1).gt(1e-6).sum().item() > 32,
      "identical inputs should get different crops/flips within one batch")
check("augment actually changes images", not torch.allclose(augment(X_train[:16]), X_train[:16]))
check("evaluation is deterministic (no augmentation / dropout / BN updates in eval)",
      evaluate(model, X_val[:2000], y_val[:2000]) == evaluate(model, X_val[:2000], y_val[:2000]))
check("initial loss ~ ln 10", abs(init_loss - math.log(10)) < 0.3, f"got {init_loss:.3f}")
check("can overfit one batch", overfit_loss < 0.05, f"got {overfit_loss:.4f}")
check("LR schedule warms up then anneals", max(lrs) > 1.5 * lrs[0] and lrs[-1] < 0.1 * max(lrs),
      f"start {lrs[0]:.3g}, max {max(lrs):.3g}, end {lrs[-1]:.3g}")
check_close("best checkpoint restored (val acc == best)", evaluate(model, X_val, y_val)[1], best_acc, atol=2e-3)
check("history per epoch", len(history["val_acc"]) == len(history["train_loss"]) > 0)
check("trained within ~4 minutes", train_seconds < 240, f"{train_seconds:.0f}s")
check("test accuracy >= 0.80", test_acc >= 0.80, f"got {test_acc:.3f}")

## Say it out loud (P3)
1. What does BatchNorm do in train vs eval mode? What breaks if you evaluate in train mode, or train with batch size 2?
2. Why is the conv bias disabled before BatchNorm?
3. Why augment only the training set? Is horizontal flip always label-preserving (think digits, text, left/right)?
4. OneCycle: what happens in each phase and why does a large peak LR help? Why step the scheduler per batch here?
5. Early stopping with OneCycle: why does it rarely trigger, and what does "keep the best checkpoint" buy instead?
6. Why can the train loss sit *above* the val loss in the plot? Is that underfitting?
7. What made this run take about a minute, and what would the bottleneck be with a standard `DataLoader` and PIL transforms?
8. How would you get from ~90 % to 94 %+ (name three things)?

In [ ]:
s.finish(lookups=0, score=None, notes="")  # score: 1-5 self-rating; notes: what tripped you up